# LifeMate Vast — notebook FINAL (từ checkpoint 1524 đến lúc nộp bài)

Một notebook duy nhất, **chạy lại nhiều lần được** (mỗi bước kiểm tra kết quả đã có rồi mới làm). Mỗi lần thuê máy:

1. Sửa **MỘT** ô cấu hình (ô 2) → *Run All*. Bước nào xong rồi sẽ được bỏ qua nhờ file trong `lifemate_runs/`.
2. Cuối notebook luôn đóng gói `lifemate-final-*.tar.gz` → **tải về trước khi huỷ máy**. Lần sau upload lại gói đó vào `/workspace` (đặt `RESUME_PACK`).

Thứ tự khuyến nghị (mỗi bước là một lần chạy riêng, đọc kết quả rồi mới sang bước sau):

| Lần | Cấu hình | Mục đích | Thời gian ước tính (RTX 3090) |
|---|---|---|---|
| 1 | `PHASES = {"asr"}` | Đo dev: ASR medium/large, có/không `--asr-correct` (không train) | ~1–1.5 giờ |
| 2 | `PHASES = {"train_noise","eval","public"}` | Train 1 epoch có ASR-noise từ 1524 rồi đo | ~7–8 giờ |
| 3 | `PHASES = {"train_final","eval","public"}` | Train trên train+dev+holdout (đo dev sẽ lạc quan) | ~9–10 giờ |
| 4 | `PHASES = {"latency"}` | Đo độ trễ để chắc ≤30 s/mẫu trước khi đóng Docker | ~20 phút |
| 0 | `PHASES = {"zs"}` | **Nên chạy trước hết.** Giữ lại 8 app khỏi train, train từ đầu 2 epoch rồi đo `dev_id` (app đã thấy) so với `dev_zs` (app chưa thấy). Public test có 11 app lạ; khoảng cách hai số này cho biết có phải mô hình nhớ schema không | ~10–11 giờ |

**Điểm cần nhớ**: dev chỉ 154 mẫu (sai số ≈ ±4 điểm); eval loss KHÔNG dự đoán điểm; công thức điểm của BTC chưa có, nên quyết định cuối dựa vào điểm public (5 lượt/ngày). Hạn Docker 16/11 23:59 (GMT+7) — cần Dockerfile từ tài liệu BTC.


In [ ]:
from pathlib import Path
import os

# ---------------- nguồn code/dữ liệu (đã dùng ở các notebook trước) ----------------
RELEASE_REPO = "Vincent-1512/ZALO_AI_CHALLENGE_2026"
RELEASE_TAG = "lifemate-resume-508-20261107"
BUNDLE_NAME = "lifemate-vast-resume-clean-508-20261107.tar.gz"
BUNDLE_SHA256 = "b7413e49c002a7afb6fea52889da370b7961a3e9c847201dac0ff0202f764bbf"
WORKSPACE_DIR = Path("/workspace")

HANDOFF_ARCHIVE = Path("/workspace/lifemate-resume-latest.tar.gz")
HANDOFF_NAME = "lifemate-resume-latest.tar.gz"
HANDOFF_SHA256 = "8849e69703fc2a62dd084b15b8ea2f1803b6fd4b8b193d2cf224518f27048b4c"
HANDOFF_RELEASE_TAG = "lifemate-resume-1524-20261109"

# Code mới (asr_correct.py, build_asr_train_data.py ...) lấy từ commit ghim cứng trên GitHub public.
CODE_COMMIT = "4767f413c72bcb0dab37f92d01ae476f7760a301"

# Gói kết quả của lần chạy trước (file lifemate-final-*.tar.gz bạn đã tải về). Upload vào /workspace rồi điền tên; "" = lần đầu.
RESUME_PACK = ""

# ---------------- chọn việc cần làm ở lần chạy này ----------------
PHASES = {"zs"}   # tập con của: "zs", "asr", "train_noise", "train_final", "eval", "public", "latency"

# ASR: các biến thể đem so trên dev (adapter 1524). Tên: (model, asr_correct)
ASR_VARIANTS = [("medium", False), ("medium", True), ("large", False), ("large", True)]
ASR_CHOICE = "auto"       # "auto" = biến thể dev state-match cao nhất NẾU hơn medium thường >= ASR_MIN_GAIN; hoặc ghi tay "medium"/"medium+correct"/"large"/"large+correct"
ASR_MIN_GAIN = 0.02       # dev nhỏ (154 mẫu) -> không đổi ASR vì chênh lệch nhỏ hơn ngưỡng này
ASR_LARGE_REPO = "vinai/PhoWhisper-large"   # ~1.55B tham số; tổng hệ thống ~5.6B < 6B, LLM 4.0B < 5B

# Train
NOISE_PROB = 0.5          # xác suất dùng transcript ASR (thay vì text gốc) khi dựng dữ liệu train
STAGE_LR = {"noise": 2e-5, "final": 2e-5}   # 2e-5 là giá trị đã đo ở warm-start (public 25.71 -> 25.99); các giá trị khác là giả thuyết
STAGE_SEED = {"noise": 2028, "final": 2029}
FINAL_INIT = "noise"      # adapter khởi tạo cho train_final: "noise" (nếu đã có) hoặc "baseline" (1524)
EVALS_PER_EPOCH = 2       # eval loss chỉ để theo dõi; early stopping tắt vì eval loss không đo điểm
SMOKE_TEST = True         # chạy thử 2 bước trước mỗi giai đoạn train mới
KEEP_CKPTS = 3            # giữ ít checkpoint để tiết kiệm đĩa; resume dùng cái mới nhất

# Public
PUBLIC_STAGES = ["noise"]            # adapter nào tạo public submission: "baseline", "noise", "final"
SKIP_PUBLIC_IF_WORSE_BY = 0.03       # chặn nếu dev state-match kém baseline hơn 3 điểm (không áp cho "final" vì dev đã lẫn vào train)
LATENCY_STAGE = "noise"
LATENCY_LIMIT_S = 25.0               # biên an toàn so với 30 s/mẫu (GPU của bạn có thể khác A30)

# ---------------- thông số run gốc (KHÔNG đổi) ----------------
RUN_ID = "vast-20261106-124816"
SEED = 2026
MAX_SEQUENCE_LENGTH = 8192
GRAD_ACCUMULATION = 16
LORA_RANK, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EVALS_PER_EPOCH_ORIG = 2
SCHEMA_PERMUTATIONS = 1
LR_ORIG = 5e-5
MAX_EPOCHS = 30

# Thí nghiệm zero-shot: giữ lại app khỏi train để đo khả năng dùng tool/app CHƯA THẤY (public test có 11 app lạ)
ZS_HELD_OUT = ["bike_rental", "door_lock", "home_automation", "home_security", "learning", "news", "robot_vacuum", "video_meeting"]
ZS_LR = 1e-4              # train TỪ ĐẦU (không warm-start) nên dùng LR cao hơn warm-start; giả thuyết
ZS_EPOCHS = 2
ZS_SEED = 2030

KNOWN = {"asr", "train_noise", "train_final", "eval", "public", "latency", "zs"}
assert PHASES <= KNOWN, PHASES - KNOWN
assert set(PUBLIC_STAGES) <= {"baseline", "noise", "final"}
assert FINAL_INIT in {"noise", "baseline"}
assert 0 < NOISE_PROB <= 1
print("PHASES =", sorted(PHASES))


## 1. Tải code + dữ liệu cơ sở, nhập checkpoint 1524, đồng bộ code mới

In [ ]:
"""Download a pinned GitHub release bundle for the Vast notebook."""
from __future__ import annotations

import hashlib
import json
import os
import shutil
import tarfile
import tempfile
import time
import urllib.error
import urllib.request
from pathlib import Path


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def api_request(url: str, token: str, binary: bool = False, offset: int = 0):
    headers = {
        "Accept": "application/octet-stream" if binary else "application/vnd.github+json",
        "User-Agent": "lifemate-vast-bootstrap",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    if offset:
        headers["Range"] = f"bytes={offset}-"
    request = urllib.request.Request(url, headers=headers)
    # urllib does not forward unredirected headers to the asset CDN.
    if token:
        request.add_unredirected_header("Authorization", f"Bearer {token}")
    return urllib.request.urlopen(request, timeout=90)


def get_release_asset(repo: str, tag: str, filename: str, token: str) -> dict:
    base = f"https://api.github.com/repos/{repo}"
    with api_request(f"{base}/releases/tags/{tag}", token) as response:
        release = json.load(response)
    assets = [asset for asset in release.get("assets", []) if asset["name"] == filename]
    if len(assets) != 1 or assets[0].get("state") != "uploaded":
        raise FileNotFoundError(f"Release {tag} chưa có asset hoàn chỉnh: {filename}")
    return assets[0]


def download_once(repo: str, asset: dict, partial: Path, token: str) -> None:
    offset = partial.stat().st_size if partial.exists() else 0
    url = f"https://api.github.com/repos/{repo}/releases/assets/{int(asset['id'])}"
    with api_request(url, token, binary=True, offset=offset) as response:
        append = response.status == 206 and offset > 0
        if append and not response.headers.get("Content-Range", "").startswith(f"bytes {offset}-"):
            raise RuntimeError("Server trả sai vị trí download tiếp tục.")
        downloaded = offset if append else 0
        last_print = downloaded
        with partial.open("ab" if append else "wb") as stream:
            while chunk := response.read(1024 * 1024):
                stream.write(chunk)
                downloaded += len(chunk)
                if downloaded - last_print >= 64 * 1024**2:
                    print(f"Downloaded {downloaded / 1024**2:.0f}/{asset['size'] / 1024**2:.0f} MiB", flush=True)
                    last_print = downloaded


def download_asset(repo: str, asset: dict, target: Path, digest: str, token: str) -> None:
    if target.is_file() and file_sha256(target) == digest:
        return
    partial = target.with_suffix(target.suffix + ".part")
    for attempt in range(3):
        try:
            if not partial.exists() or partial.stat().st_size != asset["size"]:
                download_once(repo, asset, partial, token)
            if partial.stat().st_size != asset["size"] or file_sha256(partial) != digest:
                partial.unlink(missing_ok=True)
                raise RuntimeError("Download thiếu byte hoặc sai SHA-256; sẽ tải lại.")
            partial.replace(target)
            return
        except (OSError, RuntimeError) as error:
            if attempt == 2:
                raise RuntimeError("Download chưa hoàn tất. Chạy lại cell để tiếp tục file .part.") from error
            print(f"Download bị gián đoạn ({type(error).__name__}); thử lại {attempt + 2}/3.", flush=True)
            time.sleep(2)


def extract_bundle(archive: Path, workspace: Path, digest: str) -> Path:
    project = workspace / "lifemate"
    marker = project / ".github_bundle_sha256"
    if marker.is_file() and marker.read_text().strip() == digest:
        return project
    if project.exists():
        raise FileExistsError("Đã có thư mục lifemate khác. Chọn WORKSPACE_DIR mới để giữ checkpoint hiện có.")
    with tempfile.TemporaryDirectory(prefix=".lifemate-extract-", dir=workspace) as temporary:
        with tarfile.open(archive, "r:gz") as source:
            for member in source:
                parts = Path(member.name).parts
                if not parts or parts[0] != "lifemate" or not (member.isfile() or member.isdir()):
                    raise ValueError("Bundle chứa đường dẫn hoặc loại file không được phép.")
                source.extract(member, temporary, filter="data")
        staged = Path(temporary) / "lifemate"
        if not (staged / "training/train_lora.py").is_file():
            raise FileNotFoundError("Bundle không có training/train_lora.py.")
        staged.rename(project)
    marker.write_text(digest + "\n")
    return project


def bootstrap_github_release(repo: str, tag: str, filename: str, digest: str, workspace: Path) -> Path:
    workspace.mkdir(parents=True, exist_ok=True)
    project = workspace / "lifemate"
    marker = project / ".github_bundle_sha256"
    if marker.is_file() and marker.read_text().strip() == digest:
        print("Bundle đã có trên máy; dùng project hiện tại:", project)
        return project
    if project.exists():
        raise FileExistsError("Thư mục lifemate đã tồn tại; sửa WORKSPACE_DIR thành một thư mục mới.")
    free_gib = shutil.disk_usage(workspace).free / 1024**3
    if free_gib < 22:
        raise RuntimeError(f"Chỉ còn {free_gib:.1f} GiB. Cần ít nhất 22 GiB trước khi tải và giải nén bundle.")
    token = os.environ.get("GH_TOKEN", "")
    try:
        asset = get_release_asset(repo, tag, filename, token.strip())
        archive = workspace / filename
        download_asset(repo, asset, archive, digest, token.strip())
    except urllib.error.HTTPError as error:
        raise RuntimeError(f"GitHub trả HTTP {error.code}; kiểm tra Release đã publish hoặc API đã hết lượt tải.") from None
    finally:
        token = None
    project = extract_bundle(archive, workspace, digest)
    archive.unlink()
    print("Bundle đã xác minh SHA-256 và giải nén:", project)
    return project


In [ ]:
PROJECT_DIR = bootstrap_github_release(
    RELEASE_REPO, RELEASE_TAG, BUNDLE_NAME, BUNDLE_SHA256, WORKSPACE_DIR
)
os.chdir(PROJECT_DIR)
print("Project:", PROJECT_DIR)


In [ ]:
import json, shutil, tarfile
from pathlib import PurePosixPath

if HANDOFF_RELEASE_TAG and not (HANDOFF_ARCHIVE.is_file() and file_sha256(HANDOFF_ARCHIVE) == HANDOFF_SHA256):
    try:
        asset = get_release_asset(RELEASE_REPO, HANDOFF_RELEASE_TAG, HANDOFF_NAME, "")
        download_asset(RELEASE_REPO, asset, HANDOFF_ARCHIVE, HANDOFF_SHA256, "")
    except (urllib.error.URLError, FileNotFoundError, RuntimeError, OSError) as error:
        print(f"Không tải được handoff từ Release {HANDOFF_RELEASE_TAG}: {type(error).__name__}: {error}")
        print("Sẽ dùng file upload tay nếu có.")
if not HANDOFF_ARCHIVE.is_file():
    raise FileNotFoundError(f"Upload {HANDOFF_NAME} lên {HANDOFF_ARCHIVE.parent} trước khi chạy cell này.")
if file_sha256(HANDOFF_ARCHIVE) != HANDOFF_SHA256:
    raise RuntimeError("SHA-256 của gói handoff không khớp; upload lại file.")

RUN_DIR = PROJECT_DIR / "lifemate_runs" / RUN_ID
FULL_DIR = RUN_DIR / "full"
BASE_CKPT = FULL_DIR / "checkpoint-1524"

# Chỉ lấy file của run này. Không ghi đè code trong gói (code cũ hơn); bỏ file augmentation vì sẽ tạo lại và kiểm hash.
prefix = f"lifemate/lifemate_runs/{RUN_ID}/"
skip = {prefix + "train_schema_augmented.jsonl"}
if not BASE_CKPT.exists():
    with tarfile.open(HANDOFF_ARCHIVE, "r:gz") as archive:
        members = []
        for member in archive.getmembers():
            path = PurePosixPath(member.name)
            if path.is_absolute() or ".." in path.parts:
                raise ValueError(f"Đường dẫn không an toàn trong handoff: {member.name}")
            if member.name.startswith(prefix) and member.name not in skip:
                if not (member.isfile() or member.isdir()):
                    raise ValueError(f"Loại file không được phép: {member.name}")
                members.append(member)
        archive.extractall(WORKSPACE_DIR, members=members, filter="data")

required = ["adapter_config.json", "adapter_model.safetensors", "optimizer.pt", "scheduler.pt",
            "rng_state.pth", "trainer_state.json"]
missing = [str(BASE_CKPT / name) for name in required if not (BASE_CKPT / name).is_file()]
if missing or not (FULL_DIR / "preflight.json").is_file():
    raise FileNotFoundError("Handoff thiếu file:\n" + "\n".join(missing + [str(FULL_DIR / "preflight.json")]))
state = json.loads((BASE_CKPT / "trainer_state.json").read_text())
if state.get("global_step") != 1524:
    raise RuntimeError(f"Checkpoint không đúng bước 1524: {state.get('global_step')}")

# Bản sao lưu nằm ngoài full/ nên không bị save_total_limit xoá.
BACKUP_CKPT = RUN_DIR / "backup_checkpoint-1524"
if not BACKUP_CKPT.exists():
    shutil.copytree(BASE_CKPT, BACKUP_CKPT)
print("Checkpoint 1524 OK; đã sao lưu tại", BACKUP_CKPT)


In [ ]:
import shutil, subprocess, tempfile

# Lấy code từ commit ghim cứng; chỉ chép thư mục code, KHÔNG đụng data/, models/, lifemate_runs/.
SYNC_DIRS = ("src", "training", "scripts", "configs", "tests")
SYNC_FILES = ("requirements-train.txt", "requirements-inference.txt")
stamp_file = PROJECT_DIR / ".code_commit"
if stamp_file.is_file() and stamp_file.read_text().strip() == CODE_COMMIT:
    print("Code đã đúng commit", CODE_COMMIT[:10])
else:
    tmp = Path(tempfile.mkdtemp(prefix="lifemate-code-"))
    git = lambda *a: subprocess.run(["git", *a], cwd=tmp, check=True, capture_output=True, text=True).stdout.strip()
    git("init", "-q")
    git("fetch", "--depth", "1", f"https://github.com/{RELEASE_REPO}.git", CODE_COMMIT)
    git("checkout", "-q", "FETCH_HEAD")
    head = git("rev-parse", "HEAD")
    if head != CODE_COMMIT:
        raise RuntimeError(f"Commit không khớp: {head} != {CODE_COMMIT}")
    for name in SYNC_DIRS:
        dest = PROJECT_DIR / name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(tmp / name, dest, ignore=shutil.ignore_patterns("__pycache__"))
    for name in SYNC_FILES:
        shutil.copy2(tmp / name, PROJECT_DIR / name)
    shutil.rmtree(tmp)
    stamp_file.write_text(CODE_COMMIT + "\n")
    print("Đã đồng bộ code từ commit", CODE_COMMIT[:10])
assert (PROJECT_DIR / "src/lifemate/asr_correct.py").is_file(), "thiếu asr_correct.py: CODE_COMMIT sai nhánh?"


In [ ]:
import tarfile
from pathlib import PurePosixPath

# Khôi phục kết quả lần trước (adapter các giai đoạn, báo cáo dev, cache ASR, submission).
if RESUME_PACK:
    pack_path = WORKSPACE_DIR / RESUME_PACK
    if not pack_path.is_file():
        raise FileNotFoundError(f"Upload {RESUME_PACK} vào {WORKSPACE_DIR} trước.")
    sidecar = pack_path.with_name(pack_path.name + ".sha256")
    if sidecar.is_file() and file_sha256(pack_path) != sidecar.read_text().split()[0]:
        raise RuntimeError("SHA-256 của gói kết quả không khớp.")
    with tarfile.open(pack_path, "r:gz") as archive:
        members = []
        for member in archive.getmembers():
            path = PurePosixPath(member.name)
            if path.is_absolute() or ".." in path.parts or not (member.isfile() or member.isdir()):
                raise ValueError(f"Mục không an toàn trong gói: {member.name}")
            members.append(member)
        archive.extractall(WORKSPACE_DIR, members=members, filter="data")
    print("Đã khôi phục từ", RESUME_PACK)
else:
    print("RESUME_PACK rỗng: lần chạy đầu (hoặc cùng máy, không mất đĩa).")


## 2. Đường dẫn, GPU, thư viện, dữ liệu

In [ ]:
import hashlib, json, os, random, shutil, subprocess, sys, time

# Giữ thứ tự tool theo bảng chữ cái như baseline (không xếp theo transcript ASR).
os.environ["LIFEMATE_TOOL_RETRIEVAL"] = "off"

DATA_ROOT = PROJECT_DIR / "data/raw/train+public_test"
MODEL_PATH = PROJECT_DIR / "models/Qwen3-4B-Instruct-2507"
WARM_DIR = RUN_DIR / "warm_from_1524"
ASR_CACHE = RUN_DIR / "asr_cache.json"   # chỉ memoize transcript; không đổi kết quả
PUBLIC_DIR = RUN_DIR / "public"

def run(cmd):
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=PROJECT_DIR, check=True)

def latest_checkpoint(folder):
    required = ("adapter_config.json", "adapter_model.safetensors", "optimizer.pt",
                "scheduler.pt", "rng_state.pth", "trainer_state.json")
    found = []
    for path in folder.glob("checkpoint-*"):
        if not path.is_dir():
            continue
        try:
            step = int(path.name.rsplit("-", 1)[1])
            missing = [name for name in required if not (path / name).is_file() or (path / name).stat().st_size == 0]
            if missing:
                print(f"Bỏ checkpoint thiếu file {path.name}: {missing}", flush=True)
                continue
            state = json.loads((path / "trainer_state.json").read_text(encoding="utf-8"))
            if state.get("global_step") != step:
                print(f"Bỏ checkpoint step không khớp {path.name}: global_step={state.get('global_step')}", flush=True)
                continue
            found.append((step, path))
        except (ValueError, OSError, json.JSONDecodeError) as error:
            print(f"Bỏ checkpoint lỗi {path.name}: {type(error).__name__}: {error}", flush=True)
    return max(found, default=(None, None), key=lambda item: item[0])[1]

print("Run:", RUN_DIR)


In [ ]:
subprocess.run(["nvidia-smi"], check=True)
free_gib = shutil.disk_usage(PROJECT_DIR).free / 1024**3
print(f"Free disk: {free_gib:.1f} GiB")
if free_gib < 18:
    raise RuntimeError("Ổ đĩa còn dưới 18 GiB; cần dọn hoặc thuê instance disk lớn hơn.")

try:
    import torch
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch==2.9.1", "--index-url", "https://download.pytorch.org/whl/cu130"])
    import torch
if not torch.cuda.is_available():
    raise RuntimeError("Kernel không thấy CUDA GPU. Chọn kernel Python của /venv/main.")
props = torch.cuda.get_device_properties(0)
print(f"GPU={props.name}; VRAM={props.total_memory/1024**3:.1f} GiB")
if props.total_memory < 23 * 1024**3:
    raise RuntimeError("Cần GPU 24 GB VRAM trở lên.")

if not (DATA_ROOT / "train/train.jsonl").is_file():
    dataset_archive = PROJECT_DIR / "data/archives/train+public_test.zip"
    if not dataset_archive.is_file():
        raise FileNotFoundError(dataset_archive)
    (PROJECT_DIR / "data/raw").mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-n", "-q", str(dataset_archive), "train+public_test/*", "-d", str(PROJECT_DIR / "data/raw")], check=True)

for path in (DATA_ROOT / "train/train.jsonl", DATA_ROOT / "train/train_tools_catalog.json",
             DATA_ROOT / "public_test/public_test.jsonl", BASE_CKPT / "adapter_model.safetensors"):
    if not path.is_file():
        raise FileNotFoundError(path)
print("GPU, dataset và checkpoint: OK")


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-train.txt"], check=True)
subprocess.run([sys.executable, "scripts/download_models.py"], check=True)
subprocess.run([
    sys.executable, "training/prepare_sft.py",
    "--data-root", str(DATA_ROOT),
    "--output-dir", "data/processed/sft",
    "--seed", str(SEED),
], cwd=PROJECT_DIR, check=True)
manifest = json.loads((PROJECT_DIR / "data/processed/sft/manifest.json").read_text())
print(json.dumps(manifest, ensure_ascii=False, indent=2))


In [ ]:
TRAIN_JSONL = PROJECT_DIR / "data/processed/sft/train.jsonl"
DEV_JSONL = PROJECT_DIR / "data/processed/sft/dev.jsonl"
AUG_TRAIN_JSONL = RUN_DIR / "train_schema_augmented.jsonl"
if SCHEMA_PERMUTATIONS:
    count = 0
    with TRAIN_JSONL.open(encoding="utf-8") as src, AUG_TRAIN_JSONL.open("w", encoding="utf-8") as dst:
        for line in src:
            row = json.loads(line)
            dst.write(json.dumps(row, ensure_ascii=False) + "\n")
            messages = [dict(m) for m in row["messages"]]
            payload = json.loads(messages[1]["content"])
            tools = payload.get("available_tools", [])
            if len(tools) > 1:
                key = f"{SEED}:{row['sample_id']}".encode()
                rng = random.Random(int(hashlib.sha256(key).hexdigest()[:16], 16))
                shuffled = list(tools)
                rng.shuffle(shuffled)
                if shuffled == tools:
                    shuffled = shuffled[1:] + shuffled[:1]
                payload["available_tools"] = shuffled
                messages[1]["content"] = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))
                augmented = dict(row)
                augmented["messages"] = messages
                dst.write(json.dumps(augmented, ensure_ascii=False) + "\n")
                count += 1
    TRAIN_FOR_RUN = AUG_TRAIN_JSONL
    print(f"Schema-order variants: {count}; training rows: {sum(1 for _ in AUG_TRAIN_JSONL.open())}")
else:
    TRAIN_FOR_RUN = TRAIN_JSONL


In [ ]:
common = [
    "--train-jsonl", TRAIN_FOR_RUN, "--dev-jsonl", DEV_JSONL,
    "--model-path", MODEL_PATH,
    "--gradient-accumulation-steps", GRAD_ACCUMULATION,
    "--lora-rank", LORA_RANK, "--lora-alpha", LORA_ALPHA, "--lora-dropout", LORA_DROPOUT,
    "--evals-per-epoch", EVALS_PER_EPOCH,
]
preflight = RUN_DIR / "resume_preflight.json"
run([sys.executable, "training/train_lora.py", *common, "--seed", SEED,
     "--max-sequence-length", MAX_SEQUENCE_LENGTH, "--learning-rate", LR_ORIG,
     "--preflight-only", "--auto-context", "--preflight-report", preflight,
     "--output-dir", RUN_DIR / "preflight"])
selected_context = json.loads(preflight.read_text())["selected_context"]
print("Preflight context:", selected_context)

# Dữ liệu train/dev phải trùng SHA-256 với run gốc, nếu không thì không phải tiếp tục đúng run.
original = json.loads((FULL_DIR / "preflight.json").read_text())
current = json.loads(preflight.read_text())
for split in ("train", "dev"):
    if current["splits"][split]["data_sha256"] != original["splits"][split]["data_sha256"]:
        raise RuntimeError(f"Dữ liệu {split} khác run gốc. Dừng trước khi nạp checkpoint.")
print("Train/dev SHA-256 khớp run gốc.")


## 3. ASR và hàm đo dev

In [ ]:
ASR_PATHS = {"medium": PROJECT_DIR / "models/PhoWhisper-medium", "large": PROJECT_DIR / "models/PhoWhisper-large"}
ASR_CACHES = {"medium": ASR_CACHE, "large": RUN_DIR / "asr_cache_large.json"}   # mỗi model một cache (metadata khác nhau)

need_large = ("asr" in PHASES and any(m == "large" for m, _ in ASR_VARIANTS)) or ASR_CHOICE.startswith("large")
if need_large and not (ASR_PATHS["large"] / "config.json").is_file():
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=ASR_LARGE_REPO, local_dir=str(ASR_PATHS["large"]))
if (ASR_PATHS["large"] / "config.json").is_file():
    n_params = sum(p.stat().st_size for p in ASR_PATHS["large"].glob("*.safetensors")) / 4 / 1e9  # ước lượng fp32 -> tỉ tham số
    print(f"PhoWhisper-large tải xong (ước tính ~{n_params:.2f} B tham số nếu fp32; kiểm số thật trong config trước khi nộp).")


In [ ]:
import statistics

def dev_eval(tag, adapter, asr="medium", correct=False):
    report = RUN_DIR / f"dev_{tag}.json"
    weights = Path(adapter) / "adapter_model.safetensors"
    if report.is_file() and report.stat().st_mtime > weights.stat().st_mtime:
        print(f"Dùng lại báo cáo dev: {report.name}")
    else:
        run([sys.executable, "training/evaluate_dev.py",
             "--data-root", DATA_ROOT, "--adapter", adapter, "--limit-samples", 0,
             "--report", report, "--per-sample-log", RUN_DIR / f"dev_{tag}_samples.jsonl",
             "--asr-model", ASR_PATHS[asr], "--asr-cache", ASR_CACHES[asr],
             *(["--asr-correct"] if correct else [])])
    return json.loads(report.read_text())

def latency_summary(tag):
    log = RUN_DIR / f"dev_{tag}_samples.jsonl"
    times = sorted(json.loads(l)["total_seconds"] for l in log.open(encoding="utf-8") if l.strip())
    return {"p50": statistics.median(times), "p95": times[int(0.95 * (len(times) - 1))],
            "max": times[-1], "over_limit": sum(t > LATENCY_LIMIT_S for t in times), "n": len(times)}

def fmt(r):
    return (f"state_match={r['changed_app_state_match_rate']:.4f} ({r['state_match_count']}/{r['samples']}) "
            f"call_F1_diag={r['tool_call_f1_diagnostic']:.4f} empty_pred={r['empty_prediction_samples']} end={r['end_reason_counts']}")

def variant_name(asr, correct):
    return asr + ("+correct" if correct else "")

STAGE_ADAPTER = {"baseline": BASE_CKPT, "noise": RUN_DIR / "stage_noise/final", "final": RUN_DIR / "stage_final/final"}


## 4. Đo baseline và so sánh biến thể ASR (không train)

In [ ]:
ASR_RESULTS = {}
if "asr" in PHASES:
    for asr, correct in ASR_VARIANTS:
        name = variant_name(asr, correct)
        tag = "ckpt1524" if name == "medium" else f"ckpt1524_{name.replace('+', '_')}"
        ASR_RESULTS[name] = dev_eval(tag, BASE_CKPT, asr, correct)
        r = ASR_RESULTS[name]
        lat = latency_summary(tag)
        print(f"{name:<16} {fmt(r)}\n{'':<16} latency p50={lat['p50']:.1f}s p95={lat['p95']:.1f}s max={lat['max']:.1f}s over{LATENCY_LIMIT_S:.0f}s={lat['over_limit']}/{lat['n']}")
    print("\nCảnh báo: sai số dev ≈ ±4 điểm; chỉ đổi ASR nếu hơn medium thường >= ASR_MIN_GAIN và độ trễ vẫn an toàn.")

def pick_asr():
    if ASR_CHOICE != "auto":
        name = ASR_CHOICE
    else:
        saved = {}
        for asr, correct in ASR_VARIANTS:
            n = variant_name(asr, correct)
            tag = "ckpt1524" if n == "medium" else f"ckpt1524_{n.replace('+', '_')}"
            f = RUN_DIR / f"dev_{tag}.json"
            if f.is_file():
                saved[n] = json.loads(f.read_text())["changed_app_state_match_rate"]
        base = saved.get("medium")
        if base is None:
            name = "medium"
        else:
            best = max(saved, key=saved.get)
            name = best if saved[best] - base >= ASR_MIN_GAIN else "medium"
    asr, _, corr = name.partition("+")
    return asr, bool(corr), name

SEL_ASR, SEL_CORRECT, SEL_NAME = pick_asr()
print("ASR dùng cho các bước sau:", SEL_NAME)


In [ ]:
BASE_REPORT = dev_eval("ckpt1524", BASE_CKPT, "medium", False) if PHASES & {"eval", "public", "train_noise", "train_final", "latency"} else None
SEL_TAG_SUFFIX = SEL_NAME.replace("+", "_")
print("Baseline 1524 (medium):", fmt(BASE_REPORT) if BASE_REPORT else "không đo")


## 5. Train (mỗi giai đoạn tự resume; early stopping tắt vì eval loss không đo điểm)

In [ ]:
def build_stage_data(stage):
    out = RUN_DIR / f"data_{stage}"
    target = out / "train.jsonl"
    if not target.is_file():
        cmd = [sys.executable, "scripts/build_asr_train_data.py", "--data-root", DATA_ROOT, "--output-dir", out,
               "--asr-model", ASR_PATHS[SEL_ASR], "--asr-cache", ASR_CACHES[SEL_ASR],
               "--asr-probability", NOISE_PROB, "--seed", SEED]
        if stage == "final":
            cmd.append("--include-dev-holdout")
        run(cmd)
    return target

def train_stage(stage, init_adapter):
    out = RUN_DIR / f"stage_{stage}"
    if (out / "final/adapter_config.json").is_file():
        print(f"stage_{stage}/final đã có; bỏ qua train.")
        return out / "final"
    data = build_stage_data(stage)
    rows = sum(1 for _ in data.open(encoding="utf-8"))
    steps = -(-rows // GRAD_ACCUMULATION)
    print(f"stage {stage}: {rows} bản ghi -> {steps} bước (1 epoch), LR={STAGE_LR[stage]}, init={init_adapter}")
    stage_common = list(common)
    stage_common[stage_common.index("--train-jsonl") + 1] = data
    base_cmd = [sys.executable, "training/train_lora.py", *stage_common, "--seed", STAGE_SEED[stage],
                "--max-sequence-length", selected_context, "--learning-rate", STAGE_LR[stage]]
    if SMOKE_TEST and not (RUN_DIR / f".smoke_{stage}_ok").exists():
        smoke = RUN_DIR / f"smoke_{stage}"
        if smoke.exists():
            shutil.rmtree(smoke)
        run([*base_cmd, "--epochs", 1, "--max-steps", 2, "--limit-train-samples", 8, "--limit-dev-samples", 4,
             "--evals-per-epoch", 1, "--disable-early-stopping", "--no-load-best", "--save-total-limit", 1,
             "--output-dir", smoke, "--warm-start-adapter", init_adapter])
        if not (smoke / "final/adapter_config.json").is_file():
            raise RuntimeError(f"Smoke {stage} không tạo adapter: {smoke}")
        shutil.rmtree(smoke)
        (RUN_DIR / f".smoke_{stage}_ok").write_text("ok\n")
        print("Smoke OK.")
    interrupted = latest_checkpoint(out)
    if interrupted is None and out.exists() and list(out.glob("checkpoint-*")):
        raise FileNotFoundError(f"Checkpoint dở ở {out} nhưng không đủ file để resume; giữ nguyên để kiểm tra.")
    start = ["--resume-from-checkpoint", interrupted] if interrupted else ["--warm-start-adapter", init_adapter]
    run([*base_cmd, "--epochs", MAX_EPOCHS, "--max-steps", steps, "--evals-per-epoch", EVALS_PER_EPOCH,
         "--disable-early-stopping", "--no-load-best", "--save-total-limit", KEEP_CKPTS,
         "--output-dir", out, *start])
    if not (out / "final/adapter_config.json").is_file():
        raise FileNotFoundError(f"Train xong nhưng thiếu {out / 'final'}")
    return out / "final"


In [ ]:
if "train_noise" in PHASES:
    train_stage("noise", BASE_CKPT)
else:
    print("Bỏ train_noise.")


In [ ]:
if "train_final" in PHASES:
    init = STAGE_ADAPTER["noise"] if FINAL_INIT == "noise" else BASE_CKPT
    if not (Path(init) / "adapter_config.json").is_file():
        raise FileNotFoundError(f"FINAL_INIT={FINAL_INIT} nhưng chưa có adapter {init}; chạy train_noise trước hoặc đặt FINAL_INIT='baseline'.")
    train_stage("final", init)
else:
    print("Bỏ train_final.")


## 6. Đo dev các giai đoạn

In [ ]:
STAGE_REPORTS = {}
if "eval" in PHASES:
    for stage in ("noise", "final"):
        adapter = STAGE_ADAPTER[stage]
        if (adapter / "adapter_config.json").is_file():
            STAGE_REPORTS[stage] = dev_eval(f"stage_{stage}_{SEL_TAG_SUFFIX}", adapter, SEL_ASR, SEL_CORRECT)
    print("baseline(1524, medium)", fmt(BASE_REPORT))
    for stage, r in STAGE_REPORTS.items():
        note = "  [dev đã lẫn vào train -> số lạc quan, KHÔNG so với baseline]" if stage == "final" else ""
        print(f"{stage:<22}", fmt(r), note)
    print("\nDev chỉ 154 mẫu (±4 điểm). Quyết định cuối: điểm public.")
else:
    print("Bỏ eval.")


## 7. Public inference (có chốt chặn)

In [ ]:
PUBLIC_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS = {}
if "public" in PHASES:
    public_input = DATA_ROOT / "public_test/public_test.jsonl"
    expected_ids = [json.loads(l)["id"] for l in public_input.open(encoding="utf-8") if l.strip()]
    for stage in PUBLIC_STAGES:
        adapter = STAGE_ADAPTER[stage]
        if not (adapter / "adapter_config.json").is_file():
            print(f"Bỏ {stage}: chưa có adapter {adapter}")
            continue
        rep = STAGE_REPORTS.get(stage)
        if stage == "noise" and rep and BASE_REPORT and SKIP_PUBLIC_IF_WORSE_BY is not None:
            if rep["changed_app_state_match_rate"] < BASE_REPORT["changed_app_state_match_rate"] - SKIP_PUBLIC_IF_WORSE_BY:
                print(f"CHẶN public {stage}: dev kém baseline hơn {SKIP_PUBLIC_IF_WORSE_BY:.2f}. Đặt SKIP_PUBLIC_IF_WORSE_BY=None nếu vẫn muốn nộp.")
                continue
        asr_name, correct = ("medium", False) if stage == "baseline" else (SEL_ASR, SEL_CORRECT)
        sub = PUBLIC_DIR / f"submission-{stage}-{variant_name(asr_name, correct).replace('+', '_')}-{time.strftime('%Y%m%d-%H%M%S')}.jsonl"
        run([sys.executable, "scripts/run_inference.py", "--data", DATA_ROOT, "--split", "public_test", "--output", sub,
             "--adapter", adapter, "--asr-model", ASR_PATHS[asr_name], "--asr-cache", ASR_CACHES[asr_name],
             *(["--asr-correct"] if correct else [])])
        rows = [json.loads(l) for l in sub.open(encoding="utf-8") if l.strip()]
        ids = [r.get("id") for r in rows]
        if len(rows) != len(expected_ids) or set(ids) != set(expected_ids) or len(set(ids)) != len(ids):
            raise RuntimeError("ID/số dòng submission không khớp public test.")
        if any(set(r) != {"id", "tool_calls", "state_hash"} for r in rows):
            raise RuntimeError("Submission sai schema.")
        SUBMISSIONS[stage] = sub
        print(f"Submission {stage}: {sub.name} ({len(rows)} dòng, rỗng={sum(1 for r in rows if not r['tool_calls'])})")
else:
    print("Bỏ public.")


## 8. Đo độ trễ

In [ ]:
if "latency" in PHASES:
    adapter = STAGE_ADAPTER[LATENCY_STAGE]
    tag = f"stage_{LATENCY_STAGE}_{SEL_TAG_SUFFIX}"
    dev_eval(tag, adapter, SEL_ASR, SEL_CORRECT)
    lat = latency_summary(tag)
    print(f"GPU: {subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True).stdout.strip()}")
    print(f"Độ trễ/mẫu (ASR+LLM+tool): p50={lat['p50']:.1f}s p95={lat['p95']:.1f}s max={lat['max']:.1f}s; vượt {LATENCY_LIMIT_S:.0f}s: {lat['over_limit']}/{lat['n']}")
    print("Giới hạn BTC là 30 s/mẫu trên A30 (chậm hơn 3090 một chút). Nếu max gần/vượt 30 s: giảm max_new_tokens, giảm ngữ cảnh hoặc bỏ asr-correct/large.")
else:
    print("Bỏ latency.")


## 9. Thí nghiệm zero-shot (app chưa thấy)

In [ ]:
ZS_DIR = RUN_DIR / "zero_shot"
ZS_DATA = PROJECT_DIR / "data/processed/zero_shot"
ZS_OUT = ZS_DIR / "train_run"
if "zs" in PHASES:
    if not (ZS_DATA / "manifest.json").is_file():
        run([sys.executable, "training/prepare_zero_shot_split.py", "--data-root", DATA_ROOT, "--output-dir", ZS_DATA,
             "--seed", SEED, "--held-out-apps", *ZS_HELD_OUT])
    print(json.dumps(json.loads((ZS_DATA / "manifest.json").read_text())["samples"], ensure_ascii=False))
    ZS_TRAIN_AUG = ZS_DIR / "train_schema_augmented.jsonl"
    ZS_DIR.mkdir(parents=True, exist_ok=True)
    if not ZS_TRAIN_AUG.is_file():
        with (ZS_DATA / "train.jsonl").open(encoding="utf-8") as src, ZS_TRAIN_AUG.open("w", encoding="utf-8") as dst:
            for line in src:
                row = json.loads(line)
                dst.write(json.dumps(row, ensure_ascii=False) + "\n")
                payload = json.loads(row["messages"][1]["content"])
                tools = payload.get("available_tools", [])
                if len(tools) > 1:
                    rng = random.Random(int(hashlib.sha256(f"{SEED}:{row['sample_id']}".encode()).hexdigest()[:16], 16))
                    shuffled = list(tools); rng.shuffle(shuffled)
                    if shuffled == tools:
                        shuffled = shuffled[1:] + shuffled[:1]
                    payload["available_tools"] = shuffled
                    aug = dict(row); aug["messages"] = [dict(m) for m in row["messages"]]
                    aug["messages"][1]["content"] = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))
                    dst.write(json.dumps(aug, ensure_ascii=False) + "\n")
    zs_cmd = [sys.executable, "training/train_lora.py", "--train-jsonl", ZS_TRAIN_AUG, "--dev-jsonl", ZS_DATA / "dev_id.jsonl",
              "--model-path", MODEL_PATH, "--gradient-accumulation-steps", GRAD_ACCUMULATION,
              "--lora-rank", LORA_RANK, "--lora-alpha", LORA_ALPHA, "--lora-dropout", LORA_DROPOUT,
              "--evals-per-epoch", EVALS_PER_EPOCH, "--seed", ZS_SEED, "--max-sequence-length", selected_context,
              "--learning-rate", ZS_LR, "--epochs", ZS_EPOCHS, "--disable-early-stopping", "--no-load-best",
              "--save-total-limit", KEEP_CKPTS, "--output-dir", ZS_OUT]
    if (ZS_OUT / "final/adapter_config.json").is_file():
        print("zero_shot/train_run/final đã có; bỏ qua train.")
    else:
        interrupted = latest_checkpoint(ZS_OUT) if ZS_OUT.exists() else None
        if interrupted is None and ZS_OUT.exists() and list(ZS_OUT.glob("checkpoint-*")):
            raise FileNotFoundError(f"Checkpoint dở ở {ZS_OUT} nhưng không đủ file để resume; giữ nguyên để kiểm tra.")
        run([*zs_cmd, *(["--resume-from-checkpoint", interrupted] if interrupted else [])])
    ZS_ADAPTER = ZS_OUT / "final"
    ZS_REPORTS = {}
    for name in ("dev_id", "dev_zs"):
        report = RUN_DIR / f"dev_zs_{name}.json"
        if not (report.is_file() and report.stat().st_mtime > (ZS_ADAPTER / "adapter_model.safetensors").stat().st_mtime):
            run([sys.executable, "training/evaluate_dev.py", "--data-root", DATA_ROOT, "--adapter", ZS_ADAPTER,
                 "--sft-dev", ZS_DATA / f"{name}.jsonl", "--limit-samples", 0, "--report", report,
                 "--per-sample-log", RUN_DIR / f"dev_zs_{name}_samples.jsonl",
                 "--asr-model", ASR_PATHS["medium"], "--asr-cache", ASR_CACHES["medium"]])
        ZS_REPORTS[name] = json.loads(report.read_text())
    print("dev_id (app đã thấy)  :", fmt(ZS_REPORTS["dev_id"]))
    print("dev_zs (app CHƯA thấy):", fmt(ZS_REPORTS["dev_zs"]))
    gap = ZS_REPORTS["dev_id"]["changed_app_state_match_rate"] - ZS_REPORTS["dev_zs"]["changed_app_state_match_rate"]
    print(f"\nKhoảng cách tổng quát hoá (state-match): {gap:+.3f}")
    print("Lớn (>~0.15): mô hình nhớ schema -> cần augmentation đổi tên tool/tham số và dữ liệu tổng hợp cho app lạ.")
    print("Nhỏ: nguyên nhân điểm public thấp nằm chỗ khác (ASR, ngày giờ, phân phối yêu cầu).")
else:
    print("Bỏ zs.")


## 10. Đóng gói kết quả (luôn chạy)

In [ ]:
import tarfile

stamp = time.strftime("%Y%m%d-%H%M%S")
pack = WORKSPACE_DIR / f"lifemate-final-{stamp}.tar.gz"
include = [p for p in RUN_DIR.glob("dev_*") if p.is_file()]
include += [p for p in RUN_DIR.glob("asr_cache*.json") if p.is_file()]
include += [p for p in PUBLIC_DIR.glob("submission-*.jsonl")]
for stage in ("noise", "final"):
    folder = RUN_DIR / f"stage_{stage}"
    if (folder / "final").exists():
        include.append(folder / "final")
    newest = latest_checkpoint(folder) if folder.exists() else None
    if newest is not None:
        include.append(newest)
    if (folder / "preflight.json").is_file():
        include.append(folder / "preflight.json")
zs_final = RUN_DIR / "zero_shot/train_run/final"
if zs_final.exists():
    include.append(zs_final)
if (PROJECT_DIR / "data/processed/zero_shot/manifest.json").is_file():
    include.append(PROJECT_DIR / "data/processed/zero_shot/manifest.json")
for stage in ("noise", "final"):
    d = RUN_DIR / f"data_{stage}" / "manifest.json"
    if d.is_file():
        include.append(d)
with tarfile.open(pack, "w:gz") as archive:
    for path in include:
        archive.add(path, arcname=str(Path("lifemate") / path.relative_to(PROJECT_DIR)))
digest = hashlib.sha256(pack.read_bytes()).hexdigest()
(pack.parent / (pack.name + ".sha256")).write_text(f"{digest}  {pack.name}\n")
print("Gói kết quả:", pack, f"({pack.stat().st_size / 1024**2:.0f} MiB)")
print("SHA-256:", digest)
print("TẢI file .tar.gz VÀ .sha256 về trước khi huỷ máy; lần sau điền tên vào RESUME_PACK.")


## Hoàn tất

Sau mỗi lần chạy: nộp file `submission-*.jsonl` mới lên trang public (5 lượt/ngày), ghi điểm vào bảng so sánh, rồi quyết định bước tiếp theo.

**Trước 16/11 23:59 (GMT+7):** cần Dockerfile (offline, đọc `/data`, ghi `result/submission.jsonl`, chạy tất định, ≤30 s/mẫu trên A30). Gửi tài liệu hướng dẫn Docker của BTC để viết `Dockerfile` song song với việc train. Chỉ lần nộp Docker cuối cùng được tính.
